# Harmonic downgrading: synthetic power-law benchmark

This notebook compares five approaches to downgrading a HEALPix map from
NSIDE 256 → 32, using a synthetic power-law sky with a known reference at
the target resolution.  It is designed to support a discussion about the
interface and defaults of `healpy.harmonic_ud_grade`.

## Background: Planck 2015 XVI Eq. 1

The harmonic downgrade prescription from *Planck 2015 results XVI* is

$$
a^{\mathrm{out}}_{\ell m}
= \frac{p^{\mathrm{out}}_\ell}{p^{\mathrm{in}}_\ell}
  \;\frac{b^{\mathrm{out}}_\ell}{b^{\mathrm{in}}_\ell}
  \;a^{\mathrm{in}}_{\ell m}
$$

where $p_\ell$ is the pixel window function and $b_\ell$ is a Gaussian
beam transfer function.

| # | Method | What it does (in Eq. 1 terms) | `healpy` call |
|---|--------|-------------------------------|---------------|
| 1 | **ud_grade** | Pixel averaging – no harmonic step | `hp.ud_grade(m, nside_out)` |
| 2 | **Harmonic clip** | SHT truncation only ($p=1, b=1$) | `hp.harmonic_ud_grade(m, nside_out, pixwin=False, fwhm_out=0)` |
| 3 | **Harmonic + pixwin** | Pixel-window correction ($b=1$) | `hp.harmonic_ud_grade(m, nside_out, pixwin=True, fwhm_out=0)` |
| 4 | **Harmonic + pixwin + beam** | Full Eq. 1 with default beam | `hp.harmonic_ud_grade(m, nside_out)` *(defaults: pixwin=True, fwhm_out=auto)* |
| 5 | **Smoothing + ud_grade** | Traditional smoothing then averaging | `hp.smoothing(m, fwhm=…) + hp.ud_grade()` |

**Note:** At NSIDE 32 the Planck FWHM convention is 320′ ≈ 5.33°.
Our default `3 × nside2resol(32)` ≈ 330′ ≈ 5.5°, which matches well.

In [ ]:
import numpy as np
import healpy as hp
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams.update({'font.size': 11, 'axes.grid': True, 'grid.alpha': 0.3})

COLORS = {
    'ud_grade':   '#d95f02',
    'clip':       '#1b9e77',
    'pixwin':     '#33a02c',
    'full':       '#1f78b4',
    'smooth_ud':  '#7570b3',
}
METHOD_LABELS = {
    'ud_grade':   'ud_grade',
    'clip':       'harmonic (clip)',
    'pixwin':     'harmonic + pixwin',
    'full':       'harmonic + pixwin + beam',
    'smooth_ud':  'smoothing + ud_grade',
}
METHOD_ORDER = ['ud_grade', 'clip', 'pixwin', 'full', 'smooth_ud']
LINESTYLES = {
    'ud_grade':   '-',
    'clip':       '--',
    'pixwin':     '-.',
    'full':       '-',
    'smooth_ud':  ':',
}

In [ ]:
nside_in  = 256
nside_out = 32
lmax_in   = 3 * nside_in - 1
lmax_out  = 3 * nside_out - 1

# Power-law spectrum: C_ℓ ∝ (ℓ/80)^{-2.7}
ell = np.arange(lmax_in + 1, dtype=float)
cl  = np.zeros_like(ell)
cl[2:] = (ell[2:] / 80.0) ** (-2.7)

np.random.seed(1234)
alm_in  = hp.synalm(cl, lmax=lmax_in)
m_in    = hp.alm2map(alm_in, nside_in, verbose=False)
alm_ref = hp.almxfl(alm_in, np.ones(lmax_out + 1))  # truncate
m_ref   = hp.alm2map(alm_ref, nside_out, verbose=False)

# ----- Five downgrade methods -----
m_ud   = hp.ud_grade(m_in, nside_out=nside_out)
m_clip = hp.harmonic_ud_grade(m_in, nside_out, pixwin=False, fwhm_out=0)
m_pw   = hp.harmonic_ud_grade(m_in, nside_out, pixwin=True,  fwhm_out=0)
m_full = hp.harmonic_ud_grade(m_in, nside_out)  # defaults: pixwin=True, fwhm_out=auto
fwhm_smooth = 3 * hp.nside2resol(nside_out)
m_smooth_ud = hp.ud_grade(
    hp.smoothing(m_in, fwhm=fwhm_smooth, verbose=False),
    nside_out=nside_out,
)

print(f"Input  : NSIDE={nside_in}, lmax={lmax_in}")
print(f"Output : NSIDE={nside_out}, lmax={lmax_out}")
print(f"Auto beam FWHM: {np.degrees(fwhm_smooth):.2f}° "
      f"({np.degrees(fwhm_smooth)*60:.0f}′, Planck convention: 320′)")

## Mollweide maps

In [ ]:
maps_all  = [m_ref, m_ud, m_clip, m_pw, m_full, m_smooth_ud]
titles_all = ['Reference (NSIDE 32)', METHOD_LABELS['ud_grade'],
              METHOD_LABELS['clip'], METHOD_LABELS['pixwin'],
              METHOD_LABELS['full'], METHOD_LABELS['smooth_ud']]

vals = np.concatenate([m[~np.isnan(m)] for m in maps_all])
vmin, vmax = np.percentile(vals, [1, 99])
vmin, vmax = np.round(vmin, 2), np.round(vmax, 2)

fig = plt.figure(figsize=(18, 8))
for i, (m, title) in enumerate(zip(maps_all, titles_all)):
    hp.projview(m, coord='G', cmap='RdBu_r', min=vmin, max=vmax,
                title=title,
                sub=(2, 3, i + 1), fig=fig, projection_type='mollweide',
                cbar=False)
fig.suptitle(f'Downgraded maps (NSIDE {nside_in} → {nside_out}), '
             f'shared colour scale [{vmin:.2f}, {vmax:.2f}]',
             fontsize=14, y=1.02)
plt.show()

## Residual maps (method − reference)

Residuals reveal where each method deviates from the ground-truth
reference map (synthesized directly at NSIDE 32 from the same realisation).

In [ ]:
methods_maps = [m_ud, m_clip, m_pw, m_full, m_smooth_ud]
residuals = [m - m_ref for m in methods_maps]

rvals = np.concatenate([r[~np.isnan(r)] for r in residuals])
rlim  = np.round(np.percentile(np.abs(rvals), 99), 4)

fig = plt.figure(figsize=(18, 8))
for i, (r, key) in enumerate(zip(residuals, METHOD_ORDER)):
    hp.projview(r, coord='G', cmap='RdBu_r', min=-rlim, max=rlim,
                title=f'{METHOD_LABELS[key]} − ref', unit='residual',
                sub=(2, 3, i + 1), fig=fig, projection_type='mollweide')
fig.suptitle('Residuals (method − reference)', fontsize=14, y=1.02)
plt.show()

## Gnomonic zoom

Close-ups at the **north Galactic pole** (high latitude, smooth sky) and
the **Galactic centre** (steep gradients) expose pixel-scale artefacts.
At NSIDE 32 the pixel size is ≈ 1.8°.

In [ ]:
locations = [
    ('North Galactic pole', {'rot': (0, 90)}),
    ('Galactic centre',     {'rot': (0, 0)}),
]
all_maps_gnom   = [m_ref, m_ud, m_clip, m_pw, m_full, m_smooth_ud]
all_labels_gnom = ['Reference'] + [METHOD_LABELS[k] for k in METHOD_ORDER]

for loc_name, kw in locations:
    # Compute shared colour limits for this row
    patches = []
    for m in all_maps_gnom:
        p = hp.gnomview(m, **kw, reso=10, xsize=200,
                        return_projected_map=True, no_plot=True)
        patches.append(p)
    pvals = np.concatenate([np.asarray(p)[~np.isnan(np.asarray(p))].ravel()
                            for p in patches])
    gmin, gmax = np.percentile(pvals, [1, 99])
    gmin, gmax = np.round(gmin, 3), np.round(gmax, 3)

    fig, axes = plt.subplots(1, 6, figsize=(24, 4))
    for ax, p, label in zip(axes, patches, all_labels_gnom):
        im = ax.imshow(np.asarray(p), cmap='RdBu_r', vmin=gmin, vmax=gmax,
                       origin='lower')
        ax.set_title(label, fontsize=9)
        ax.axis('off')
    fig.suptitle(f'Gnomonic zoom — {loc_name}', fontsize=13)
    fig.tight_layout()
    plt.show()

## Power spectra (pixel-window deconvolved)

All measured spectra are divided by the output pixel window
$p_\ell^2(N_{\mathrm{side,out}})$ so they show the underlying spectral
shape each method recovers.  The dashed black line is the input theory
$C_\ell$.

In [ ]:
pw_out = hp.pixwin(nside_out, lmax=lmax_out)
pw2    = pw_out ** 2

ell_out = np.arange(lmax_out + 1)

cl_ref  = hp.anafast(m_ref,  lmax=lmax_out) / pw2
cl_maps = {}
for key, m in zip(METHOD_ORDER,
                  [m_ud, m_clip, m_pw, m_full, m_smooth_ud]):
    cl_maps[key] = hp.anafast(m, lmax=lmax_out) / pw2

fig, ax = plt.subplots(figsize=(10, 6))
ax.loglog(ell_out[2:], cl[2:lmax_out+1], 'k--', lw=1.5, label='Input theory $C_\\ell$')
ax.loglog(ell_out[2:], cl_ref[2:], 'k-', lw=2, alpha=0.4, label='Reference')
for key in METHOD_ORDER:
    ax.loglog(ell_out[2:], cl_maps[key][2:],
              color=COLORS[key], ls=LINESTYLES[key], lw=1.4,
              label=METHOD_LABELS[key])
ax.set_xlabel(r'Multipole $\ell$')
ax.set_ylabel(r'$C_\ell / p_\ell^2$')
ax.set_title('Pixel-window-deconvolved power spectra')
ax.legend(fontsize=9)
ax.set_xlim(2, lmax_out)
fig.tight_layout()
plt.show()

## Fractional spectral difference

$(C_\ell^{\mathrm{method}} - C_\ell^{\mathrm{ref}}) / C_\ell^{\mathrm{ref}}$,
where both numerator and denominator are pixel-window-deconvolved.
The grey dotted curve shows $p_\ell^2 - 1$, indicating where pixel
smoothing suppresses power.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

for ax, xlim, title in [(ax1, (2, lmax_out), 'Full range'),
                         (ax2, (lmax_out//2, lmax_out), r'High-$\ell$ zoom')]:
    for key in METHOD_ORDER:
        frac = (cl_maps[key] - cl_ref) / cl_ref
        ax.plot(ell_out[2:], frac[2:],
                color=COLORS[key], ls=LINESTYLES[key], lw=1.4,
                label=METHOD_LABELS[key])
    ax.plot(ell_out[2:], (pw2[2:] - 1), ':',
            color='grey', lw=1.2, label=r'$p_\ell^2 - 1$')
    ax.axhline(0, color='k', lw=0.5)
    ax.set_xlabel(r'Multipole $\ell$')
    ax.set_ylabel(r'$(C_\ell^{\mathrm{method}} - C_\ell^{\mathrm{ref}})'
                  r' / C_\ell^{\mathrm{ref}}$')
    ax.set_title(title)
    ax.set_xlim(xlim)
    ax.legend(fontsize=8, loc='best')

fig.suptitle('Fractional spectral difference relative to reference',
             fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

## Takeaways

| Observation | Implication for defaults |
|---|---|
| **ud_grade** shows aliased power that contaminates all scales | A harmonic method is always preferable |
| **Clip-only** matches reference well at low ℓ, but the sharp bandlimit cutoff causes ringing visible in gnomview | Pixel window correction is needed |
| **Pixwin-only** fixes the spectral bias but still has a sharp cutoff | An output beam suppresses ringing |
| **Full Eq. 1** (pixwin + beam) gives the smoothest maps with excellent spectral recovery up to the beam scale | Recommended as default |
| Our auto-FWHM (3 × θ_pix ≈ 330′ at NSIDE 32) matches the Planck convention (320′) | Good default choice |
| **Smoothing + ud_grade** performs similarly to full Eq. 1 but doesn't deconvolve the input pixel window | Full Eq. 1 is more principled |